## **Random Forest Cultivation**
### Experimenting with Feature Selection to Improve Performance
---

## **Overview**

This notebook attempts to improve upon the best-performing model which was **Random Forest** with an accuracy of **0.7517** by experimenting with feature selection.

Since not all features contribute equally and some may be introducing noise, we iteratively remove the least important features one at a time based on its feature importances. It aims to improve importance while getting a much leaner feature set.

The same best hyperparameters (n_estimators=500, max_depth=None, bootstrap=Truue) from the previous notebook are reused throughout, so any change in performance can be attributed solely to the change in features.


In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

In [2]:
# Restore variables from ThimkersProcessing
%store -r

print("Variables restored successfully!")
print(f"\nDataset shapes:")
print(f"Training: {X_train_scaled.shape}")
print(f"Validation: {X_val_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"\nBest Random Forest:")
print(f"Best hyperparameters: {best_rf_label}")
print(f"n_estimators: {best_n}")
print(f"max_depth: {best_depth}")
print(f"bootstrap: {best_bootstrap}")
print(f"Baseline accuracy: {acc_rf:.4f}")

Variables restored successfully!

Dataset shapes:
Training: (29514, 395)
Validation: (9838, 395)
Test: (9839, 395)

Best Random Forest:
Best hyperparameters: depth=20/bootstrap=False
n_estimators: 500
max_depth: 20
bootstrap: False
Baseline accuracy: 0.7513


## **Experiment: Iterative Feature Elimination**
### Strategy: Remove least important features one by one and track performance
---

In [3]:
# Get feature names and initial feature importances
feature_names = X_clean.columns.tolist()
n_features = len(feature_names)

# Get initial importances from the baseline model
initial_importances = rf_best.feature_importances_

print(f"Total features: {n_features}")

Total features: 395


In [4]:
# Iterative Feature Elimination Loop
# Start with all features, remove least important one at a time

X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)
X_val_df = pd.DataFrame(X_val_scaled, columns=feature_names)
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Track results
n_features_list = []
train_errors = []
val_errors = []
test_errors = []
removed_features = []
accuracies = []
precisions = []

# Start with all features
current_features = feature_names.copy()

for iteration in range(n_features):
    n_current = len(current_features)
    
    # Train Random Forest with current feature set
    rf = RandomForestClassifier(
        n_estimators=best_n,
        max_depth=best_depth,
        bootstrap=best_bootstrap,
        random_state=42,
        n_jobs=-1
    )
    
    X_train_current = X_train_df[current_features]
    X_val_current = X_val_df[current_features]
    X_test_current = X_test_df[current_features]
    
    rf.fit(X_train_current, y_train)
    
    train_acc = rf.score(X_train_current, y_train)
    val_acc = rf.score(X_val_current, y_val)
    test_acc = rf.score(X_test_current, y_test)
    train_precision = precision_score(y_train, rf.predict(X_train_current))
    val_precision = precision_score(y_val, rf.predict(X_val_current))
    test_precision = precision_score(y_test, rf.predict(X_test_current))

    n_features_list.append(n_current)
    train_errors.append(1 - train_acc)
    val_errors.append(1 - val_acc)
    test_errors.append(1 - test_acc)
    accuracies.append(test_acc)
    precisions.append(test_precision)
    
    print(f"Features: {n_current:>3} | Train Err: {1-train_acc:.4f} | Val Err: {1-val_acc:.4f} | Test Err: {1-test_acc:.4f} | Test Acc: {test_acc:.4f} | Test Prec: {test_precision:.4f}")
    
    # Find least important feature and remove it
    if n_current > 150:
        importances = rf.feature_importances_
        least_important_index = np.argmin(importances)
        least_important_feature = current_features[least_important_index]
        removed_features.append(least_important_feature)
        current_features.pop(least_important_index)
    else:
        break

Features: 395 | Train Err: 0.0629 | Val Err: 0.2547 | Test Err: 0.2487 | Test Acc: 0.7513 | Test Prec: 0.7042
Features: 394 | Train Err: 0.0633 | Val Err: 0.2534 | Test Err: 0.2479 | Test Acc: 0.7521 | Test Prec: 0.7072
Features: 393 | Train Err: 0.0628 | Val Err: 0.2565 | Test Err: 0.2486 | Test Acc: 0.7514 | Test Prec: 0.7027
Features: 392 | Train Err: 0.0636 | Val Err: 0.2547 | Test Err: 0.2503 | Test Acc: 0.7497 | Test Prec: 0.6970
Features: 391 | Train Err: 0.0630 | Val Err: 0.2537 | Test Err: 0.2489 | Test Acc: 0.7511 | Test Prec: 0.7007
Features: 390 | Train Err: 0.0629 | Val Err: 0.2549 | Test Err: 0.2513 | Test Acc: 0.7487 | Test Prec: 0.6943
Features: 389 | Train Err: 0.0629 | Val Err: 0.2558 | Test Err: 0.2512 | Test Acc: 0.7488 | Test Prec: 0.6914
Features: 388 | Train Err: 0.0636 | Val Err: 0.2542 | Test Err: 0.2502 | Test Acc: 0.7498 | Test Prec: 0.6975
Features: 387 | Train Err: 0.0627 | Val Err: 0.2553 | Test Err: 0.2471 | Test Acc: 0.7529 | Test Prec: 0.7030
Features: 

In [5]:
# Plot Errors
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=n_features_list, y=train_errors,
    mode='lines+markers', name='Train Error',
    line=dict(color='royalblue', width=2),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=n_features_list, y=val_errors,
    mode='lines+markers', name='Validation Error',
    line=dict(color='orange', width=2),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=n_features_list, y=test_errors,
    mode='lines+markers', name='Test Error',
    line=dict(color='green', width=2),
    marker=dict(size=4)
))

# Mark the best validation error point
best_val_idx = np.argmin(val_errors)
best_n_features = n_features_list[best_val_idx]
best_val_error = val_errors[best_val_idx]

fig.add_vline(
    x=best_n_features, 
    line_dash='dash', 
    line_color='red',
    annotation_text=f'Best: {best_n_features} features<br>Val Error: {best_val_error:.4f}',
    annotation_position='top right'
)

fig.update_layout(
    title='Random Forest - Error Rate vs Number of Features<br>(Iterative Least Important Feature Elimination)',
    xaxis_title='Number of Features',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=600,
    hovermode='x unified'
)

fig.update_xaxes(autorange='reversed')  # Reverse so we go from many to few features

fig.show()

print(f"\n{'='*60}")
print(f"RESULTS SUMMARY")
print(f"{'='*60}")
print(f"Baseline (all {n_features} features):")
print(f"  Train Error: {train_errors[0]:.4f}")
print(f"  Val Error:   {val_errors[0]:.4f}")
print(f"  Test Error:  {test_errors[0]:.4f}")
print(f"\nBest Configuration ({best_n_features} features):")
print(f"  Train Error: {train_errors[best_val_idx]:.4f}")
print(f"  Val Error:   {val_errors[best_val_idx]:.4f}")
print(f"  Test Error:  {test_errors[best_val_idx]:.4f}")
print(f"\nImprovement:")
print(f"  Val Error Change:  {val_errors[0] - best_val_error:+.4f}")
print(f"  Test Error Change: {test_errors[0] - test_errors[best_val_idx]:+.4f}")
print(f"  Features Removed:  {n_features - best_n_features}")


RESULTS SUMMARY
Baseline (all 395 features):
  Train Error: 0.0629
  Val Error:   0.2547
  Test Error:  0.2487

Best Configuration (163 features):
  Train Error: 0.0456
  Val Error:   0.2441
  Test Error:  0.2420

Improvement:
  Val Error Change:  +0.0107
  Test Error Change: +0.0067
  Features Removed:  232


## **Analysis: Top Removed Features**
### Let's see which features were eliminated first
---

In [6]:
print("First 30 Features Removed (Least Important):")
print("="*60)
for i, feature in enumerate(removed_features[:30], 1):
    print(f"{i:>3}. {feature}")

n_remaining = min(30, len(removed_features))
print(f"\n\nLast {n_remaining} Features Removed (Most Important):")
print("="*60)
for i, feature in enumerate(removed_features[-n_remaining:], 1):
    print(f"{i:>3}. {feature}")

First 30 Features Removed (Least Important):
  1. ai_orchestration_martian
  2. ai_orchestration_smol_agi
  3. ai_observe_metero
  4. aimodel_reka_flash_3_or_other_reka_models
  5. ai_knowledge_letta
  6. ai_orchestration_phidata
  7. ai_observe_helicone
  8. ai_observe_opik
  9. region_central_asia
 10. region_caribbean
 11. ai_external_openhands_formerly_opendevin
 12. ai_orchestration_lyzr
 13. ai_orchestration_agno
 14. region_middle_africa
 15. ai_external_glean_enterprise_agents
 16. ai_observe_adversarial_robustness_toolbox_art
 17. ai_knowledge_zep
 18. ai_observe_protect_ai
 19. ai_knowledge_lancedb
 20. lang_mojo
 21. ai_observe_vectra_ai
 22. ai_observe_galileo
 23. aimodel_cohere:_command_a
 24. ai_observe_arize
 25. ai_knowledge_weaviate
 26. ai_knowledge_mem0
 27. db_datomic
 28. ai_orchestration_smolagents
 29. ai_knowledge_fireproof
 30. ai_external_devin_ai


Last 30 Features Removed (Most Important):
  1. devtype_manager
  2. db_bigquery
  3. devenv_webstorm
  4. plat

## **Final Model Performance**
---

In [7]:
#Best model accuracy and precision

best_test_acc = max(accuracies)
best_test_precision = max(precisions)

print(f"Best Test Accuracy: {best_test_acc:.4f}")
print(f"Best Test Precision: {best_test_precision:.4f}")

Best Test Accuracy: 0.7606
Best Test Precision: 0.7095
